In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# 경기 일정(예정 경기) 크롤링

아직 안 열린(점수가 없는) 경기만 골라서 대회명/일시/구장/라운드/양팀 이름을 `schedule_{YEAR}.xlsx`로 저장한다. 기존 박스스코어 크롤러(`KBSA_크롤링_...`)와는 완전히 별개 파이프라인이다 — 안 열린 경기는 타석 기록이 없어서 season_full로 합칠 수 없고, Streamlit의 "경기 일정" 미리보기 전용으로만 쓴다.

In [ ]:
import os
import re
import time
import random
import requests
import socket
from urllib.parse import urlparse
import pandas as pd
from datetime import date, timedelta
from bs4 import BeautifulSoup


YEAR = 2026
ROOT_DIR = f"/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
FULL_TRACKED_PATH = f"{ROOT_DIR}/season_full/full_tracked_{YEAR}.xlsx"
SCHEDULE_OUT = f"{ROOT_DIR}/season_full/schedule_{YEAR}.xlsx"
SCORE_BOX_OUT = f"{ROOT_DIR}/season_full/score_box_{YEAR}.xlsx"  # 일시정지 경기의 중단 시점 스코어보드
KIND_CD = 31  # 18세 이하부(고교야구)
CALENDAR_URL = "https://www.korea-baseball.com/game/calendar?kind_cd={kind_cd}&year={year}&month={month}"  # month는 제로패딩 금지 — 이 사이트는 "08"을 못 알아듣고 현재 달로 폴백한다. "8"로 넘겨야 그 달이 나온다.
BOX_SCORE_URL = "https://www.korea-baseball.com/game/box_score?game_idx={game_idx}"

TODAY = date.today()
START_DATE = TODAY - timedelta(days=1)  # 어제부터 — 자정 근처 경기 등 하루 여유를 둔다.
SLEEP_RANGE = (9, 11)                   # box_score 요청 간격. 37경기 기준 6~7분.


def ensure_dns(host):
    """Colab 런타임의 DNS가 이 호스트를 못 풀 때가 있어, 공용 DoH 로 A레코드를 직접 받아
    getaddrinfo 에 심어둔다(SNI·인증서 검증은 그대로 유지된다).

    반환값: 우회로 잡아둔 IP. 시스템 리졸버가 멀쩡하거나 DoH 도 실패하면 None.
    """
    overrides = getattr(socket, "_dns_override", None)
    if overrides is None:
        overrides = socket._dns_override = {}
        _orig_getaddrinfo = socket.getaddrinfo

        def _getaddrinfo(h, port, *args, **kwargs):
            if h in socket._dns_override:
                return [(socket.AF_INET, socket.SOCK_STREAM, 6, "",
                         (socket._dns_override[h], port))]
            return _orig_getaddrinfo(h, port, *args, **kwargs)

        socket.getaddrinfo = _getaddrinfo

    if host in overrides:
        return overrides[host]

    try:
        socket.getaddrinfo(host, 443, socket.AF_INET, socket.SOCK_STREAM)
        return None
    except socket.gaierror:
        pass

    ip = None
    for resolver in ("https://8.8.8.8/resolve", "https://1.1.1.1/dns-query"):
        try:
            r = requests.get(resolver,
                             params={"name": host, "type": "A"},
                             headers={"accept": "application/dns-json"},
                             timeout=10)
            ip = next((a["data"] for a in r.json().get("Answer", [])
                       if a.get("type") == 1), None)
        except Exception:
            ip = None
        if ip:
            break
    if not ip:
        return None

    overrides[host] = ip
    return ip


def get_with_retry(url, max_retries=5, label=""):
    """DNS 우회를 걸어가며 재시도하는 GET. 끝내 실패하면 마지막 예외를 그대로 올린다."""
    for attempt in range(max_retries):
        try:
            return requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        except requests.exceptions.ConnectionError as e:
            if attempt >= max_retries - 1:
                raise e
            ip = ensure_dns(urlparse(url).hostname)
            wait = 2 if ip else 5 * (attempt + 1)
            detail = f", DoH로 {ip} 확보" if ip else ""
            print(f"  {label}연결 실패 ({attempt+1}/{max_retries}){detail}, {wait}초 후 재시도...")
            time.sleep(wait)


def target_months(today=TODAY):
    """조회할 달을 (year, month) 쌍으로 반환한다 — 지금은 이번 달 하나만 본다.

    예전엔 이번 달 + 다음 달을 같이 훑었지만, 요청 수를 아끼려고 이번 달로 줄였다.
    대신 다음 달로 순연된 경기는 그 달이 될 때까지 일정 목록에 안 잡힌다 —
    시즌 막바지처럼 다음 달 경기까지 봐야 하면 next_month를 다시 넣으면 된다.

    주의: month 파라미터는 제로패딩하면 안 된다 — CALENDAR_URL에서 month={month}로
    넘긴다. 예전엔 month={month:02d}로 "08"을 보내다 보니 사이트가 그 값을 무시하고
    현재 달로 폴백해서 '두 달이 같은 목록을 준다'고 오해했는데, 실제로는 "8"처럼
    넘기면 각 달이 정상적으로 다르게 나온다. game_id 중복 제거는 만일을 위해 그대로 둔다.
    """
    return [(today.year, today.month)]


def load_full_tracked_game_ids(path=FULL_TRACKED_PATH):
    """full_tracked_{YEAR}.xlsx(주자상황 체크.ipynb가 저장 -> 스트림릿 제조.ipynb가
    읽는 파일)에 이미 들어있는 game_id 전체를 가져온다. 여기 이미 있는 game_id는
    정식으로 다 처리된 경기라는 뜻이라, 예정 목록에 남아있으면 안 된다."""
    # game_id 한 열만 있으면 되는데 xlsx 는 8만행을 통째로 파싱해야 한다(11초). 통합
    # 체크가 같이 저장하는 parquet 는 열 하나만 뽑아 읽을 수 있어 사실상 즉시 끝난다.
    parq = os.path.splitext(path)[0] + '.parquet'
    if os.path.exists(parq):
        df = pd.read_parquet(parq, columns=['game_id'])
    elif os.path.exists(path):
        df = pd.read_excel(path, usecols=['game_id'])
    else:
        return set()
    return set(pd.to_numeric(df['game_id'], errors='coerce').dropna().astype(int).tolist())


def fetch_calendar_games(year, month, kind_cd=KIND_CD):
    """캘린더 페이지에서 그 달의 모든 경기(game_idx, 날짜)를 뽑는다.

    이 페이지가 주는 건 game_idx·일시·점수뿐이다. 팀명은 잘려 보일 수 있고
    (예: "서울아이티") 구장·대회·라운드는 아예 없다 — 그래서 여기서는 '어떤
    경기가 있는가'만 얻고, 표시에 쓰는 값은 전부 box_score에서 받는다.

    종료 여부로 거르지 않는다: 캘린더엔 종료로 보이지만 우리 파이프라인이 아직
    수집 못 한 경기(결과 크롤링 실패분)가 일정에도 결과에도 안 남는 사각지대가
    생기기 때문이다. 제외는 full_tracked(정말로 수집 완료된 game_id)로만 한다.

    li의 date 속성("2026:07:11:09:00")에서 날짜를 뽑아 같이 담는다 —
    START_DATE(어제) 이전 경기를 호출하는 쪽에서 걸러내는 데 쓴다."""
    url = CALENDAR_URL.format(kind_cd=kind_cd, year=year, month=month)
    res = get_with_retry(url, label=f"{year}년 {month}월 캘린더 ")
    res.encoding = "utf-8"
    soup = BeautifulSoup(res.text, "html.parser")

    games = []
    for li in soup.select("li[date]"):
        a = li.find("a", href=re.compile(r"game_idx=\d+"))
        if not a:
            continue
        m = re.search(r"game_idx=(\d+)", a["href"])
        if not m:
            continue

        date_tokens = li.get("date", "").split(":")
        game_date = date(*map(int, date_tokens[:3])) if len(date_tokens) >= 3 else None
        games.append({"game_id": m.group(1), "game_date": game_date})
    return games


def parse_score_box(soup, game_idx, team1, team2):
    """box_score 의 이닝별 스코어보드(원정/홈 두 행)를 그대로 뽑는다.

    "각_game별_아웃카운트_및_점수_추출.ipynb" 의 parse_inning_summary 와 같은 표를
    읽지만, 그쪽은 우리 기록지와 대조하는 검수용(inning_summary)이고 이건 기록지가
    아예 없는 일시정지 경기를 사이트에 띄우기 위한 스냅샷이라 파일을 나눠 둔다.

    이닝 칸이 두 번 연달아 비면 거기서 끊는다 — 중단된 경기는 그 뒤가 통째로 비어 있다.
    한 칸만 비는 경우(홈팀이 그 이닝을 아직 안 침)는 None 으로 자리를 남긴다."""
    table = soup.select_one(".tb_head table") or soup.select_one(".box_table table")
    if not table:
        return None
    rows = table.select("tbody tr")
    if len(rows) < 2:
        return None

    def _int(cell):
        txt = re.sub(r"[^\d]", "", cell.get_text(strip=True))
        return int(txt) if txt else 0

    out = {"game_id": int(game_idx), "team1": team1, "team2": team2}
    for idx, row in enumerate(rows[:2], start=1):
        cells = row.find_all(["th", "td"])
        if len(cells) < 6:
            return None
        inning_cells, rheb = cells[1:-4], cells[-4:]

        runs, empty = [], 0
        for cell in inning_cells:
            if cell.get_text(strip=True) == "":
                empty += 1
                if empty >= 2:
                    break
                runs.append(None)
                continue
            runs.append(_int(cell))
        while runs and runs[-1] is None:
            runs.pop()

        out[f"runs{idx}"] = ",".join("" if r is None else str(r) for r in runs)
        out[f"score{idx}"] = _int(rheb[0])
        out[f"H{idx}"] = _int(rheb[1])
        out[f"E{idx}"] = _int(rheb[2])
        out[f"B{idx}"] = _int(rheb[3])
    return out


def save_score_box(new_rows, tracked_ids, path=SCORE_BOX_OUT):
    """일시정지 경기의 스코어보드를 score_box_{YEAR}.xlsx 에 누적 저장한다.

    이번에 다시 받은 경기는 덮어쓰고(중단 시점 스코어가 나중에 정정될 수 있다),
    이번 조회에 안 잡힌 예전 경기는 그대로 둔다 — 캘린더 조회 범위(어제 이후)를
    벗어났다는 이유로 사이트에서 경기가 통째로 사라지면 안 되기 때문이다.
    재개돼서 full_tracked 에 들어간 경기는 이제 정상 경기라 여기서 지운다
    (웹은 그때부터 season_full 쪽 기록을 쓴다)."""
    prev = pd.read_excel(path) if os.path.exists(path) else pd.DataFrame()
    df = pd.DataFrame(new_rows)
    new_ids = set(df["game_id"].astype(int).tolist()) if not df.empty else set()

    if not prev.empty:
        prev["game_id"] = pd.to_numeric(prev["game_id"], errors="coerce").astype("Int64")
        prev = prev[prev["game_id"].notna() & ~prev["game_id"].isin(new_ids)]
    combined = pd.concat([prev, df], ignore_index=True) if not df.empty else prev

    if not combined.empty:
        combined["game_id"] = pd.to_numeric(combined["game_id"], errors="coerce").astype("Int64")
        resumed = combined[combined["game_id"].isin(tracked_ids)]
        if len(resumed):
            print(f"  재개 완료로 목록에서 제외: {sorted(resumed['game_id'].dropna().astype(int).tolist())}")
        combined = combined[~combined["game_id"].isin(tracked_ids)]

    print()
    if combined.empty:
        if os.path.exists(path):
            cols = list(prev.columns) if not prev.empty else ["game_id"]
            pd.DataFrame(columns=cols).to_excel(path, index=False)
            print("일시정지 경기 없음 — score_box 를 비웠습니다.")
        else:
            print("일시정지 경기 없음 — score_box 파일 미생성")
        return combined

    if "date" in combined.columns:
        sort_cols = ["date", "time"] if "time" in combined.columns else ["date"]
        combined = combined.sort_values(sort_cols)
    combined = combined.reset_index(drop=True)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    combined.to_excel(path, index=False)
    print(f"일시정지 경기 {len(combined)}건 -> {path}")
    for _, r in combined.iterrows():
        print(f"  {int(r['game_id'])} {r.get('team1')} {r.get('score1')}:{r.get('score2')} {r.get('team2')}"
              f" ({r.get('date')} · {r.get('runs1')} / {r.get('runs2')})")
    return combined


def fetch_game_detail(game_idx, max_retries=3, retry_wait=30):
    """box_score 페이지에서 대회명/일시/구장/라운드/양 팀 전체 이름/상태를 뽑는다.
    아직 안 열린 경기도, 취소된 경기도 이 페이지 자체는 존재해서 그대로 쓸 수
    있다 — .sum_top > dl.game_name 이 "2026.07.11 09:00 / 목동야구장 / 리그전"
    처럼 원래 예정된 일정 정보를 담고 있고, 취소된 경기는 .hero_sum 안에 별도로
    "[경기취소]" dl이 하나 더 있다(정상/예정 경기엔 없음)."""
    url = BOX_SCORE_URL.format(game_idx=game_idx)

    res = None
    for attempt in range(max_retries):
        try:
            res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
            break
        except requests.exceptions.ConnectionError as e:
            if attempt < max_retries - 1:
                ip = ensure_dns(urlparse(url).hostname)
                wait = 2 if ip else retry_wait
                detail = f", DoH로 {ip} 확보" if ip else ""
                print(f"  [{game_idx}] 연결 실패 ({attempt+1}/{max_retries}){detail}, {wait}초 후 재시도...")
                time.sleep(wait)
            else:
                print(f"  [{game_idx}] 연결 실패, 재시도 초과: {e}")
                return None

    res.encoding = "utf-8"
    soup = BeautifulSoup(res.text, "html.parser")

    game_name = soup.find("dl", class_="game_name")
    if not game_name:
        return None
    dt_el = game_name.find("dt")
    dd_el = game_name.find("dd")
    tournament = dt_el.get_text(strip=True) if dt_el else None

    dd_parts = [p.strip() for p in dd_el.get_text(strip=True).split("/")] if dd_el else []
    date_time = dd_parts[0] if len(dd_parts) > 0 else ""
    stadium = dd_parts[1] if len(dd_parts) > 1 else ""
    league = dd_parts[2] if len(dd_parts) > 2 else ""
    date_str, _, time_str = date_time.partition(" ")

    hero = soup.find("div", class_="hero_sum")
    team1 = team2 = None
    status = "예정"
    score_box = None
    if hero:
        cancel_dl = hero.find("dl", class_="game_name")
        if cancel_dl and "취소" in cancel_dl.get_text(strip=True):
            status = "취소"
        t1 = hero.select_one(".team1 .team dt .txt")
        t2 = hero.select_one(".team2 .team dt .txt")
        team1 = t1.get_text(strip=True) if t1 else None
        team2 = t2.get_text(strip=True) if t2 else None

        # 일시정지(서스펜디드) 경기 — box_score 에선 .btn_result 에 "일시정지경기" 라벨이
        # 뜬다(record_detail 에선 a.btn_rboxbk_big165_gray, PBP 크롤러의 is_game_suspended
        # 가 보는 게 그것). 이 경기는 재개 전까지 기록지가 안 올라오고 PBP 크롤러가
        # 걸러내므로 inning_summary/batter_box/pitcher_box 도 자동으로 안 긁힌다.
        # 대신 중단 시점 스코어보드만 여기서 받아 score_box 로 넘긴다 — 사이트 표시용.
        btn = hero.find("div", class_="btn_result")
        label = btn.get_text(strip=True) if btn else ""
        if "일시정지" in label:
            status = "일시정지"
            score_box = parse_score_box(soup, game_idx, team1, team2)
            if score_box:
                score_box.update({
                    "date": date_str, "time": time_str, "stadium": stadium,
                    "league": league, "tournament": tournament, "특이사항": label,
                })
            else:
                print(f"  [{game_idx}] 일시정지 경기인데 스코어보드 파싱 실패 — 사이트 구조 확인 필요")

    detail = {
        "game_id": game_idx,
        "상태": status,
        "date": date_str,
        "time": time_str,
        "stadium": stadium,
        "league": league,
        "tournament": tournament,
        "team1": team1,
        "team2": team2,
    }
    if score_box:
        # schedule 파일 열을 늘리지 않도록 호출하는 쪽에서 빼내 간다.
        detail["_score_box"] = score_box
    return detail


WATCH_FIELDS = ["date", "time", "stadium", "league", "tournament", "team1", "team2", "상태"]


def describe_changes(prev_row, detail):
    """기존 schedule 행과 이번에 받은 값이 어디가 달라졌는지 문자열 목록으로.
    바뀐 게 없으면 빈 목록. 구장 변경(목동 -> 신월)도 여기서 잡힌다."""
    if not prev_row:
        return ["신규"]
    out = []
    for f in WATCH_FIELDS:
        before, after = str(prev_row.get(f, "")).strip(), str(detail.get(f, "")).strip()
        if before in ("nan", "None"):
            before = ""
        if before != after:
            out.append(f"{f} {before or '(없음)'} -> {after or '(없음)'}")
    return out


def crawl_upcoming_schedule(year=YEAR, sleep_range=SLEEP_RANGE):
    """어제부터(START_DATE) ~ 이번 달 캘린더를 훑어서 "아직 정상 종료되지 않은"
    경기(예정 또는 취소)를 schedule_{YEAR}.xlsx로 저장한다.

    ★ 후보로 잡힌 경기는 매번 box_score를 다시 받는다.
      예전에는 '캘린더에서 점수 칸이 비어 있고 일시도 그대로면 기존 행 재사용'
      으로 요청을 아꼈다. 그런데 캘린더가 주는 건 game_idx·일시·점수뿐이라
      **구장·대회·라운드·팀명이 바뀌어도 알 방법이 없다** — 구장이 목동에서
      신월로 바뀐 경기가 옛 구장을 단 채로 계속 남았다(2026-08-07). 후보가
      40경기 안팎이라 전부 다시 받아도 6~7분이면 끝나므로, 재사용을 없애고
      정확성을 택했다. 급하면 sleep_range를 (2, 3)으로 줄이면 된다.

    날짜 하한: START_DATE(어제) 이전 날짜의 경기는 아예 대상에서 뺀다 — 오래된
    취소 경기를 계속 들고 있지 않기 위해서다. 옛날에 취소된 경기가 스케줄에
    남아있었다면, 오늘 새로 만드는 이 목록에는 안 들어가서 자연히 빠진다.

    제외 기준: full_tracked_{YEAR}.xlsx(정식 파이프라인이 이미 다 처리한 경기)에
    있는 game_id는 무조건 제외한다.

    중복 제거: 지금은 이번 달 하나만 조회하지만, target_months()를 다시 여러 달로
    넓혔을 때를 대비해 game_id 기준으로 먼저 나온 것만 남기고 중복을 지운다.

    기존 파일은 덮어쓸 값을 만드는 데 쓰지 않고 **무엇이 바뀌었는지 보여주는 용도**
    로만 읽는다(describe_changes). 결과는 언제나 이번에 새로 받은 값이다.
    """
    tracked_ids = load_full_tracked_game_ids()
    print(f"full_tracked에 이미 있는 game_id {len(tracked_ids)}개 -> 예정 목록에서 제외 대상")
    print(f"날짜 하한(어제): {START_DATE}\n")

    months = target_months()
    print(f"조회 대상 달: {months}\n")

    all_games = []
    seen_ids = set()
    for y, m in months:
        print(f"=== {y}년 {m}월 캘린더 조회 ===")
        month_games = fetch_calendar_games(y, m)
        before = len(month_games)
        month_games = [
            g for g in month_games
            if int(g["game_id"]) not in tracked_ids
            and g["game_date"] is not None and g["game_date"] >= START_DATE
        ]
        # 같은 달 안에서든 여러 달을 볼 때든, game_id가 이미 나왔으면 건너뛴다.
        new_games = []
        for g in month_games:
            if g["game_id"] in seen_ids:
                continue
            seen_ids.add(g["game_id"])
            new_games.append(g)
        print(
            f"  캘린더 기준 후보 {before}개 -> 필터링 후 {len(month_games)}개 "
            f"-> 중복 제거 후 {len(new_games)}개"
        )
        all_games.extend(new_games)
        time.sleep(random.uniform(2, 3))

    existing_by_id = {}
    if os.path.exists(SCHEDULE_OUT):
        prev_df = pd.read_excel(SCHEDULE_OUT, dtype={"game_id": str})
        prev_df = prev_df.drop_duplicates(subset=["game_id"])  # 예전에 만들어진 중복 파일 방어
        existing_by_id = {row["game_id"]: row.to_dict() for _, row in prev_df.iterrows()}
        print(f"\n기존 schedule 파일에 {len(existing_by_id)}행 있음 (변경 확인용)")

    n = len(all_games)
    est = n * (sum(sleep_range) / 2 + 1.5) / 60
    print(f"\n{n}경기 전부 box_score 재조회 — 예상 {est:.0f}분\n")

    rows, failed, changed, score_boxes = [], [], [], []
    for i, g in enumerate(all_games, 1):
        gid = g["game_id"]
        detail = fetch_game_detail(gid)
        if not detail:
            failed.append(gid)
            print(f"  [{i}/{n}] {gid} 상세 조회 실패 — 기존 행이 있으면 유지")
            if existing_by_id.get(gid):
                rows.append(existing_by_id[gid])
            time.sleep(random.uniform(*sleep_range))
            continue

        sb = detail.pop("_score_box", None)
        if sb:
            score_boxes.append(sb)
        rows.append(detail)
        diffs = describe_changes(existing_by_id.get(gid), detail)
        mark = f"  ★ {' | '.join(diffs)}" if diffs else ""
        if diffs and diffs != ["신규"]:
            changed.append((gid, diffs))
        print(
            f"  [{i}/{n}] {gid} {detail['team1']} vs {detail['team2']} "
            f"({detail['date']} {detail['time']} {detail['stadium']} · {detail['상태']}){mark}"
        )
        time.sleep(random.uniform(*sleep_range))

    schedule_df = pd.DataFrame(rows).drop_duplicates(subset=["game_id"])
    os.makedirs(os.path.dirname(SCHEDULE_OUT), exist_ok=True)
    schedule_df.to_excel(SCHEDULE_OUT, index=False)

    print(f"\n{'='*60}")
    print(f"완료: {SCHEDULE_OUT} ({len(schedule_df)}행 / 조회 {n}건)")
    save_score_box(score_boxes, tracked_ids)
    if changed:
        print(f"\n바뀐 경기 {len(changed)}건")
        for gid, diffs in changed:
            print(f"  {gid}: {' | '.join(diffs)}")
    else:
        print("\n바뀐 경기 없음")
    if failed:
        print(f"\n조회 실패 {len(failed)}건: {failed}")
    return schedule_df


# ==================== 실행 ====================
schedule_df = crawl_upcoming_schedule()
